In [1]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import time
import string
import pandas as pd
import os
import glob
import json

In [2]:
def analizza_giocatore(url, scraper):
    res = scraper.get(url)
    if res.status_code != 200:
        return None

    soup = BeautifulSoup(res.content, 'html.parser')
    info = soup.find('div', id='info')
    stats_pullout = soup.find('div', class_='stats_pullout')

    dati = {'URL': url}
    dati['Nome'] = info.find('h1').text.strip() if info and info.find('h1') else "N/D"

    if info:
        for p in info.find_all('p'):
            label = p.find('strong')
            if not label: continue

            testo_label = label.get_text()
            testo_completo = p.get_text(separator=" ", strip=True)

            if "Position:" in testo_label:
                # Estrae la stringa grezza
                raw_position = testo_completo.split("Position:")[1].split("▪")[0].strip()

                # Sostituisce ", and " e " and " con una semplice virgola
                raw_position = raw_position.replace(", and ", ", ").replace(" and ", ", ")

                # Crea la lista dividendo per virgola e rimuovendo gli spazi vuoti
                dati['Position'] = [pos.strip() for pos in raw_position.split(",") if pos.strip()]
            elif "Shoots:" in testo_label:
                dati['Shoots'] = testo_completo.split("Shoots:")[1].split("▪")[0].strip()
            elif "Team:" in testo_label:
                dati['Team'] = p.find('a').text.strip() if p.find('a') else "N/D"
            elif "Born:" in testo_label:
                span = p.find('span', id='necro-birth')
                dati['Born'] = span.get_text(separator=" ", strip=True) if span else "N/D"
            elif "Draft:" in testo_label:
                links = p.find_all('a')
                if links:
                    dati['Draft Team'] = links[0].text
                    match_anno = re.search(r'\d{4}', links[-1].text)
                    dati['Draft Year'] = match_anno.group() if match_anno else "N/D"
            elif "College:" in testo_label or "Colleges:" in testo_label:
                # Gestisce sia un singolo college che una lista di college separati da virgola
                links_college = p.find_all('a')
                dati['College'] = [a.text.strip() for a in links_college] if links_college else []
            elif "High School:" in testo_label:
                testo_scuola = testo_completo.split("High School:")[1].strip()
                dati['High School'] = testo_scuola.split(" in ")[0].strip()
            elif "NBA Debut:" in testo_label:
                anno_match = re.search(r'\d{4}', testo_completo)
                dati['Debut'] = anno_match.group() if anno_match else "N/D"
            elif "Experience:" in testo_label or "Career Length:" in testo_label:
                match_exp = re.search(r'\d+', testo_completo.split(":")[1])
                dati['Experience'] = match_exp.group() if match_exp else "0"

        # Estrazione Altezza e Peso
        match_fisico = re.search(r'\((\d+cm),\s*(\d+kg)\)', info.get_text())
        if match_fisico:
            dati['Altezza'] = match_fisico.group(1)
            dati['Peso'] = match_fisico.group(2)

    # Estrazione Statistiche di Carriera
    dati['Career Stats'] = {}
    if stats_pullout:
        for stat_block in stats_pullout.find_all('div'):
            label_stat = stat_block.find('span', class_='poptip')
            if label_stat:
                key = label_stat.text.strip()
                if key in ['G', 'PTS', 'TRB', 'AST', 'FG%', 'FG3%', 'FT%', 'eFG%', 'PER', 'WS']:
                    valori = stat_block.find_all('p')
                    if valori:
                        # Prende sempre l'ultimo <p> che corrisponde ai dati "Career"
                        dati['Career Stats'][key] = valori[-1].text.strip()

    return dati

In [3]:
scraper = cloudscraper.create_scraper(browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False})
base_url = "https://www.basketball-reference.com"
alfabeto = string.ascii_lowercase

tutti_i_link = {}

for lettera in alfabeto:
    url = f"https://www.basketball-reference.com/players/{lettera}/"

    response = scraper.get(url)
    link_estratti = []

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        player_cells = soup.find_all('th', attrs={'data-stat': 'player'})

        for cell in player_cells:
            link = cell.find('a')
            if link and 'href' in link.attrs:
                link_estratti.append(base_url + link['href'])

        tutti_i_link[lettera] = link_estratti
        print(f"Estratti {len(link_estratti)} link per la lettera '{lettera.upper()}'")
    else:
        print(f"Errore {response.status_code} per la lettera '{lettera.upper()}'")

    time.sleep(3) # Pausa di sicurezza

totale = 0
for lettera, links in tutti_i_link.items():
    totale += len(links)
print(f"\nTotale link estratti: {totale}")

Estratti 179 link per la lettera 'A'
Estratti 517 link per la lettera 'B'
Estratti 342 link per la lettera 'C'
Estratti 267 link per la lettera 'D'
Estratti 119 link per la lettera 'E'
Estratti 164 link per la lettera 'F'
Estratti 271 link per la lettera 'G'
Estratti 382 link per la lettera 'H'
Estratti 29 link per la lettera 'I'
Estratti 270 link per la lettera 'J'
Estratti 188 link per la lettera 'K'
Estratti 212 link per la lettera 'L'
Estratti 509 link per la lettera 'M'
Estratti 115 link per la lettera 'N'
Estratti 101 link per la lettera 'O'
Estratti 245 link per la lettera 'P'
Estratti 10 link per la lettera 'Q'
Estratti 274 link per la lettera 'R'
Estratti 472 link per la lettera 'S'
Estratti 215 link per la lettera 'T'
Estratti 12 link per la lettera 'U'
Estratti 61 link per la lettera 'V'
Estratti 418 link per la lettera 'W'
Estratti 0 link per la lettera 'X'
Estratti 23 link per la lettera 'Y'
Estratti 21 link per la lettera 'Z'

TOTALE LINK ESTRATTI: 5416


In [5]:
# Esecuzione di prova
urls = [
    "https://www.basketball-reference.com/players/j/jamesle01.html", "https://www.basketball-reference.com/players/j/jordaje01.html"
]

scraper = cloudscraper.create_scraper(browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False})

for link in urls:
    risultato = analizza_giocatore(link, scraper)
    print(f"\nRisultati per: {risultato.get('Nome', 'Sconosciuto')}")
    for chiave, valore in risultato.items():
        if chiave == 'Career Stats':
            print("Summary Carriera:")
            print(", ".join([f"{k}: {v}" for k, v in valore.items()]))
        else:
            print(f"{chiave}: {valore}")


Risultati per: LeBron James
URL: https://www.basketball-reference.com/players/j/jamesle01.html
Nome: LeBron James
Position: ['Small Forward', 'Power Forward', 'Point Guard', 'Center', 'Shooting Guard']
Born: December 30 , 1984
High School: St. Vincent-St. Mary
Draft Team: Cleveland Cavaliers
Draft Year: 2003
Debut: 2003
Experience: 22
Altezza: 206cm
Peso: 113kg
Summary Carriera:
G: 1622, PTS: 26.8, TRB: 7.5, AST: 7.4, FG%: 50.7, FG3%: 34.8, FT%: 73.7, eFG%: 54.9, PER: 26.7, WS: 276.8

Risultati per: Jerome Jordan
URL: https://www.basketball-reference.com/players/j/jordaje01.html
Nome: Jerome Jordan
Position: ['Center']
Born: September 29 , 1986
College: ['Tulsa']
High School: Florida Preparatory Academy
Draft Team: Milwaukee Bucks
Draft Year: 2010
Debut: 2011
Experience: 2
Altezza: 213cm
Peso: 114kg
Summary Carriera:
G: 65, PTS: 2.8, TRB: 2.0, AST: 0.3, FG%: 52.8, FG3%: -, FT%: 85.2, eFG%: 52.8, PER: 17.1, WS: 1.7


In [ ]:
# Configurazione iniziale
scraper = cloudscraper.create_scraper(browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False})
base_url = "https://www.basketball-reference.com"

lettera = 'u'
lettere_da_estrarre = [lettera]

tutti_i_urls = []

# Estrazione URL
print("Fase 1: Raccolta dei link dei giocatori...")
for lettera in lettere_da_estrarre:
    print(f"Raccolta link per la lettera '{lettera.upper()}'...")
    url_alfabeto = f"https://www.basketball-reference.com/players/{lettera}/"
    response = scraper.get(url_alfabeto)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        for cell in soup.find_all('th', attrs={'data-stat': 'player'}):
            link = cell.find('a')
            if link and 'href' in link.attrs:
                tutti_i_urls.append(base_url + link['href'])
    time.sleep(2) # Pausa tra una pagina dell'alfabeto e l'altra

print(f"Trovati {len(tutti_i_urls)} giocatori. Inizio l'estrazione delle schede...")

# Estrazione schede e costruzione dataset
dati_giocatori = []

for i, url_giocatore in enumerate(tutti_i_urls):
    print(f"Estrazione {i+1}/{len(tutti_i_urls)}: {url_giocatore}")
    dati_grezzi = analizza_giocatore(url_giocatore, scraper)

    if dati_grezzi:
        # Estrazione le statistiche dal dizionario annidato
        stats = dati_grezzi.pop('Career Stats', {})
        dati_grezzi.update(stats) # Aggiunge PTS, TRB, ecc come chiavi principali

        dati_giocatori.append(dati_grezzi)

    time.sleep(3) # Pausa per non farsi bloccare

# Creazione dataset e savlataggio
df_giocatori = pd.DataFrame(dati_giocatori)

# Salvataggio in formato json
path_file = "../data/raw/dataset_giocatori_nba_"+lettera+".json"
df_giocatori.to_json(path_file, orient="records", indent=4, force_ascii=False)

print(f"\nOperazione completata! Dataset creato con {len(df_giocatori)} giocatori")

Fase 1: Raccolta dei link dei giocatori...
Raccolta link per la lettera 'U'...
Trovati 12 giocatori. Inizio l'estrazione delle schede...
Estrazione 1/12: https://www.basketball-reference.com/players/u/ubileed01.html
Estrazione 2/12: https://www.basketball-reference.com/players/u/udohek01.html
Estrazione 3/12: https://www.basketball-reference.com/players/u/udokaim01.html
Estrazione 4/12: https://www.basketball-reference.com/players/u/udrihbe01.html
Estrazione 5/12: https://www.basketball-reference.com/players/u/ukicro01.html
Estrazione 6/12: https://www.basketball-reference.com/players/u/ulisty01.html
Estrazione 7/12: https://www.basketball-reference.com/players/u/umudest01.html
Estrazione 8/12: https://www.basketball-reference.com/players/u/unselwe01.html
Estrazione 9/12: https://www.basketball-reference.com/players/u/uplinha01.html
Estrazione 10/12: https://www.basketball-reference.com/players/u/upshake01.html
Estrazione 11/12: https://www.basketball-reference.com/players/u/uthofja01.

In [9]:
display(df_giocatori.head())

,URL,Nome,Position,Born,College,High School,Debut,Experience,Altezza,Peso,...,TRB,AST,FG%,FG3%,FT%,eFG%,PER,WS,Draft Team,Draft Year
0,https://www.basketball-reference.com/players/u...,Edwin Ubiles,[Small Forward],"November 26 , 1986",[Siena College],St. Thomas More School,2012,1,198cm,92kg,...,2.5,0.3,27.8,20.0,100.0,30.6,7.3,0.0,NaN,NaN
1,https://www.basketball-reference.com/players/u...,Ekpe Udoh,"[Center, Power Forward]","May 20 , 1987","[Michigan, Baylor]",Santa Fe,2010,7,208cm,111kg,...,2.9,0.7,45.3,0.0,71.8,45.3,12.0,10.4,Golden State Warriors,2010
2,https://www.basketball-reference.com/players/u...,Ime Udoka,"[Small Forward, Shooting Guard, Power Forward]","August 9 , 1977","[San Francisco, Portland State]",Jefferson,2004,7,198cm,97kg,...,2.9,1.0,41.7,35.6,70.5,49.2,10.8,8.8,NaN,NaN
3,https://www.basketball-reference.com/players/u...,Beno Udrih,"[Point Guard, Shooting Guard]","July 5 , 1982",NaN,NaN,2004,13,190cm,92kg,...,2.1,3.4,46.3,34.9,83.3,50.2,14.0,31.4,San Antonio Spurs,2004
4,https://www.basketball-reference.com/players/u...,Roko Ukić,[Point Guard],"May 12 , 1984",NaN,NaN,2008,2,196cm,83kg,...,0.9,1.9,38.7,18.9,74.6,40.7,10.0,-0.2,Toronto Raptors,2005


In [ ]:
# Inserisci il percorso della cartella che contiene i tuoi file JSON
percorso_cartella = '../data/raw/'

# Trova tutti i file che finiscono con ".json" all'interno della cartella
lista_file_json = glob.glob(os.path.join(percorso_cartella, "*.json"))

dati_totali = []

print(f"Trovati {len(lista_file_json)} file JSON. Inizio l'unione...")

# Cicla su ogni file, estrai il contenuto e aggiungilo alla lista principale
for file in lista_file_json:
    with open(file, 'r', encoding='utf-8') as f:
        try:
            contenuto = json.load(f)

            # Se il file contiene una lista di dizionari (es. più giocatori)
            if isinstance(contenuto, list):
                dati_totali.extend(contenuto)
            # Se il file contiene un solo dizionario (es. un singolo giocatore)
            elif isinstance(contenuto, dict):
                dati_totali.append(contenuto)

        except json.JSONDecodeError:
            print(f"Attenzione: Impossibile leggere il file {file} (Formato non valido)")

# Converte la lista gigante in un unico DataFrame Pandas
df_finale = pd.DataFrame(dati_totali)
df_finale.to_json("../data/dataset_nba_completo.json", orient="records", indent=4, force_ascii=False)

print(f"\nOperazione completata! Il DataFrame finale ha {df_finale.shape[0]} righe e {df_finale.shape[1]} colonne.")

# Mostra le prime 5 righe per verificare che i dati siano stati uniti correttamente
display(df_finale.head())

Trovati 1 file JSON. Inizio l'unione...

Operazione completata! Il DataFrame finale ha 12 righe e 22 colonne.


,URL,Nome,Position,Born,College,High School,Debut,Experience,Altezza,Peso,...,TRB,AST,FG%,FG3%,FT%,eFG%,PER,WS,Draft Team,Draft Year
0,https://www.basketball-reference.com/players/u...,Edwin Ubiles,[Small Forward],"November 26 , 1986",[Siena College],St. Thomas More School,2012,1,198cm,92kg,...,2.5,0.3,27.8,20.0,100.0,30.6,7.3,0.0,NaN,NaN
1,https://www.basketball-reference.com/players/u...,Ekpe Udoh,"[Center, Power Forward]","May 20 , 1987","[Michigan, Baylor]",Santa Fe,2010,7,208cm,111kg,...,2.9,0.7,45.3,0.0,71.8,45.3,12.0,10.4,Golden State Warriors,2010
2,https://www.basketball-reference.com/players/u...,Ime Udoka,"[Small Forward, Shooting Guard, Power Forward]","August 9 , 1977","[San Francisco, Portland State]",Jefferson,2004,7,198cm,97kg,...,2.9,1.0,41.7,35.6,70.5,49.2,10.8,8.8,NaN,NaN
3,https://www.basketball-reference.com/players/u...,Beno Udrih,"[Point Guard, Shooting Guard]","July 5 , 1982",None,NaN,2004,13,190cm,92kg,...,2.1,3.4,46.3,34.9,83.3,50.2,14.0,31.4,San Antonio Spurs,2004
4,https://www.basketball-reference.com/players/u...,Roko Ukić,[Point Guard],"May 12 , 1984",None,NaN,2008,2,196cm,83kg,...,0.9,1.9,38.7,18.9,74.6,40.7,10.0,-0.2,Toronto Raptors,2005


In [ ]:
import pandas as pd
import cloudscraper
from bs4 import BeautifulSoup
import re
import time

print("🔄 Avvio script di patching")

# Carica il dataset
percorso_file = "../data/dataset_nba_completo.json"
df = pd.read_json(percorso_file)

colonna_nome = 'Player' if 'Player' in df.columns else ('Nome' if 'Nome' in df.columns else 'player')

# Identifica i giocatori senza Debut
mask_mancanti = df['Debut'].isnull() | (df['Debut'] == "N/D") | (df['Debut'] == "")
indici_da_cercare = df[mask_mancanti].index

print(f"Trovati {len(indici_da_cercare)} giocatori senza 'Debut'. Inizio scraping mirato...\n")

# Inizializza CloudScraper
scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    }
)

giocatori_corretti = 0

# Ciclo di Scraping Mirato
for indice in indici_da_cercare:
    url = df.at[indice, 'URL']
    nome = df.at[indice, colonna_nome]

    try:
        res = scraper.get(url)

        if res.status_code == 429:
            print("❌ ERRORE 429: Rate Limiting! Hai fatto troppe richieste in poco tempo. Aumenta il time.sleep().")
            break
        elif res.status_code != 200:
            print(f"⚠️ Errore HTTP {res.status_code} caricando la pagina di {nome}")
            continue

        soup = BeautifulSoup(res.content, 'html.parser')
        info = soup.find('div', id='info')

        trovato = False
        if info:
            for p in info.find_all('p'):
                label = p.find('strong')
                if not label: continue

                testo_label = label.get_text()

                # Ricerca flessibile per NBA, ABA o BAA
                if "Debut:" in testo_label:
                    testo_completo = p.get_text(separator=" ", strip=True)
                    anno_match = re.search(r'\d{4}', testo_completo)

                    if anno_match:
                        anno = anno_match.group()
                        df.at[indice, 'Debut'] = anno
                        print(f"✅ Corretto {nome.ljust(25)} -> {anno} ({testo_label.strip()})")
                        giocatori_corretti += 1
                        trovato = True
                        break

        if not trovato:
            print(f"❓ Nessun anno di debutto trovato nella pagina di {nome}")

        # Pausa fondamentale per non farsi bannare
        time.sleep(2.5)

    except Exception as e:
        print(f"❌ Errore durante lo scraping di {nome}: {e}")

print(f"Patch Completata! Aggiornati {giocatori_corretti} record.")

🔄 Avvio script di patching con CloudScraper per i Debutti mancanti...
Trovati 429 giocatori senza 'Debut'. Inizio scraping mirato...

❓ Nessun anno di debutto trovato nella pagina di Ben Schadler


/tmp/ipykernel_691/2300557415.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1973' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[indice, 'Debut'] = anno


✅ Corretto Billy Schaeffer           -> 1973 (ABA Debut:)
❓ Nessun anno di debutto trovato nella pagina di Otto Schnellbacher
❓ Nessun anno di debutto trovato nella pagina di Milt Schoon
✅ Corretto Roger Schurig             -> 1967 (ABA Debut:)
✅ Corretto Willie Scott              -> 1969 (ABA Debut:)
✅ Corretto Paul Scranton             -> 1967 (ABA Debut:)
✅ Corretto Les Selvage               -> 1967 (ABA Debut:)
❓ Nessun anno di debutto trovato nella pagina di Frank Selvy
❓ Nessun anno di debutto trovato nella pagina di Paul Seymour
✅ Corretto Lynn Shackelford          -> 1970 (ABA Debut:)
❓ Nessun anno di debutto trovato nella pagina di Earl Shannon
❓ Nessun anno di debutto trovato nella pagina di Howie Shannon
❓ Nessun anno di debutto trovato nella pagina di Bob Shea
✅ Corretto Billy Shepherd            -> 1972 (ABA Debut:)
✅ Corretto Don Sidle                 -> 1968 (ABA Debut:)
✅ Corretto Grant Simmons             -> 1967 (ABA Debut:)
✅ Corretto Walt Simon                -> 196

In [ ]:
import pandas as pd
import cloudscraper
from bs4 import BeautifulSoup
import re
import time

print("🔄 Avvio script di patching (Bio + Fallback Statistiche)...")

colonna_nome = 'Player' if 'Player' in df.columns else ('Nome' if 'Nome' in df.columns else 'player')

# Identifica i giocatori senza Debut
mask_mancanti = df['Debut'].isnull() | (df['Debut'] == "N/D") | (df['Debut'] == "")
indici_da_cercare = df[mask_mancanti].index

print(f"Trovati {len(indici_da_cercare)} giocatori senza 'Debut'. Inizio scraping mirato...\n")

# Inizializza CloudScraper
scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    }
)

giocatori_corretti = 0

# Ciclo di Scraping Mirato
for indice in indici_da_cercare:
    url = df.at[indice, 'URL']
    nome = df.at[indice, colonna_nome]

    try:
        res = scraper.get(url)

        if res.status_code == 429:
            print("❌ ERRORE 429: Rate Limiting! Aumenta il time.sleep().")
            break
        elif res.status_code != 200:
            print(f"⚠️ Errore HTTP {res.status_code} caricando la pagina di {nome}")
            continue

        soup = BeautifulSoup(res.content, 'html.parser')
        info = soup.find('div', id='info')

        trovato = False

        # TENTATIVO 1: Ricerca nella Bio (NBA, ABA, BAA Debut)
        if info:
            for p in info.find_all('p'):
                label = p.find('strong')
                if not label: continue

                testo_label = label.get_text()

                if "Debut:" in testo_label:
                    testo_completo = p.get_text(separator=" ", strip=True)
                    anno_match = re.search(r'\d{4}', testo_completo)

                    if anno_match:
                        anno = anno_match.group()
                        df.at[indice, 'Debut'] = anno
                        print(f"✅ Bio: Corretto {nome.ljust(25)} -> {anno} ({testo_label.strip()})")
                        giocatori_corretti += 1
                        trovato = True
                        break

        # TENTATIVO 2 (FALLBACK): Estrazione dalla tabella "Per Game"
        if not trovato:
            tabella_stat = soup.find('table', id='per_game_stats')
            if tabella_stat:
                corpo_tabella = tabella_stat.find('tbody')
                if corpo_tabella:
                    # Prende la prima riga della carriera
                    prima_riga = corpo_tabella.find('tr')
                    if prima_riga:
                        # Cerca la cella della stagione
                        cella_anno = prima_riga.find(['th', 'td'], {'data-stat': 'year_id'})
                        if cella_anno:
                            testo_anno = cella_anno.get_text(strip=True) # Es. "1947-48"
                            # Estrae i primi 4 numeri in assoluto (il 1947)
                            anno_match = re.search(r'^(\d{4})', testo_anno)

                            if anno_match:
                                anno = anno_match.group(1)
                                df.at[indice, 'Debut'] = anno
                                print(f"✅ Tabella: Corretto {nome.ljust(22)} -> {anno} (Estratto da {testo_anno})")
                                giocatori_corretti += 1
                                trovato = True

        if not trovato:
            print(f"❓ Nessun debutto trovato nella pagina di {nome} (nemmeno nelle tabelle)")

        time.sleep(2.5)

    except Exception as e:
        print(f"❌ Errore durante lo scraping di {nome}: {e}")

print(f" Patch completata! Aggiornati {giocatori_corretti} record.")


🔄 Avvio script di patching avanzato (Bio + Fallback Statistiche)...
Trovati 129 giocatori senza 'Debut'. Inizio scraping mirato...



/tmp/ipykernel_691/4115542098.py:90: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1947' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[indice, 'Debut'] = anno


✅ Tabella: Corretto Ben Schadler           -> 1947 (Estratto da 1947-48)
✅ Tabella: Corretto Otto Schnellbacher     -> 1948 (Estratto da 1948-49)
✅ Tabella: Corretto Milt Schoon            -> 1946 (Estratto da 1946-47)
✅ Tabella: Corretto Frank Selvy            -> 1954 (Estratto da 1954-55)
✅ Tabella: Corretto Paul Seymour           -> 1947 (Estratto da 1947-48)
✅ Tabella: Corretto Earl Shannon           -> 1946 (Estratto da 1946-47)
✅ Tabella: Corretto Howie Shannon          -> 1948 (Estratto da 1948-49)
✅ Tabella: Corretto Bob Shea               -> 1946 (Estratto da 1946-47)
✅ Tabella: Corretto Belus Smawley          -> 1946 (Estratto da 1946-47)
✅ Tabella: Corretto Don Smith              -> 1947 (Estratto da 1947-48)
✅ Tabella: Corretto Odie Spears            -> 1948 (Estratto da 1948-49)
✅ Tabella: Corretto Art Spector            -> 1946 (Estratto da 1946-47)
✅ Tabella: Corretto Lou Spicer             -> 1946 (Estratto da 1946-47)
✅ Tabella: Corretto Jim Springer           -> 1948 

In [ ]:
import pandas as pd
import cloudscraper
from bs4 import BeautifulSoup
import time

print("🔄 Avvio Patching delle High School...")

colonna_nome = 'Player' if 'Player' in df.columns else ('Nome' if 'Nome' in df.columns else 'player')

# Funzione di conversione: Trasforma tutto in liste
def converti_in_lista(val):
    # Se è già una lista, lascia così
    if isinstance(val, list):
        return val
    # Se è nullo, vuoto, "N/D" o "Non frequentata", restituisci lista vuota
    if pd.isnull(val) or val == "N/D" or val == "Non frequentata" or str(val).strip() == "":
        return []
    # Se è una stringa normale, la metti in una lista
    return [str(val).strip()]

print("✅ Conversione di tutti i record in liste in corso...")
df['High School'] = df['High School'].apply(converti_in_lista)

# Identifica chi ha la lista vuota e ha bisogno dello scraping
mask_mancanti = df['High School'].apply(lambda x: len(x) == 0)
indici_da_cercare = df[mask_mancanti].index

print(f"Trovati {len(indici_da_cercare)} giocatori senza High School. Inizio scraping mirato...\n")

# Inizializza CloudScraper
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

giocatori_corretti = 0

# Ciclo di Scraping
for indice in indici_da_cercare:
    url = df.at[indice, 'URL']
    nome = df.at[indice, colonna_nome]

    try:
        res = scraper.get(url)

        if res.status_code == 429:
            print("❌ ERRORE 429: Rate Limiting! Aumenta il time.sleep().")
            break
        elif res.status_code != 200:
            print(f"⚠️ Errore {res.status_code} su {nome}")
            continue

        soup = BeautifulSoup(res.content, 'html.parser')
        info = soup.find('div', id='info')

        trovato = False
        if info:
            for p in info.find_all('p'):
                label = p.find('strong')
                if not label: continue

                testo_label = label.get_text()

                # Cerca sia il singolare che il plurale
                if "High School:" in testo_label or "High Schools:" in testo_label:
                    scuole = []
                    for node in p.contents:
                        if isinstance(node, str):
                            testo = node.strip()
                            if " in " in testo:
                                testo_pulito = testo.lstrip(",").strip()
                                nome_scuola = testo_pulito.split(" in ")[0].strip()
                                if nome_scuola:
                                    scuole.append(nome_scuola)

                    if scuole:
                        df.at[indice, 'High School'] = scuole
                        print(f"✅ Corretto {nome.ljust(25)} -> {scuole}")
                        giocatori_corretti += 1
                        trovato = True
                    break

        if not trovato:
            print(f"❓ Nessuna High School trovata per {nome} (Probabilmente Internazionale/Non frequentata)")

        time.sleep(2.5)

    except Exception as e:
        print(f"❌ Errore durante lo scraping di {nome}: {e}")

# Salvataggio del dataset aggiornato
path_file= "../data/dataset_nba_completo.json"
df.to_json(path_file, orient="records", indent=4, force_ascii=False)

print(f"Patch completata! Recuperate {giocatori_corretti} scuole.")


🔄 Avvio Patching delle High School...
✅ Conversione di tutti i record in liste in corso...
Trovati 806 giocatori senza High School. Inizio scraping mirato...

❓ Nessuna High School trovata per Guerschon Yabusele (Probabilmente Internazionale/Non frequentata)
❓ Nessuna High School trovata per Barry Yates (Probabilmente Internazionale/Non frequentata)
✅ Corretto Jahmir Young              -> ["St. Mary's Ryken", 'DeMatha Catholic']
✅ Corretto Korleone Young            -> ['Wichita East', 'Hargrave Military Academy']
✅ Corretto Sam Young                 -> ['Friendly', 'Hargrave Military Academy']
❓ Nessuna High School trovata per Sun Yue (Probabilmente Internazionale/Non frequentata)
❓ Nessuna High School trovata per Omer Yurtseven (Probabilmente Internazionale/Non frequentata)
❓ Nessuna High School trovata per Arvydas Sabonis (Probabilmente Internazionale/Non frequentata)
❓ Nessuna High School trovata per Domantas Sabonis (Probabilmente Internazionale/Non frequentata)
❓ Nessuna High Scho